In [2]:
# scripts/bronze_dlt_pipeline.py
import dlt
from dlt.sources.filesystem import filesystem
import pyarrow as pa
import sys
import os

notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))

import pytz
import pandas as pd
import duckdb
from my_schemas.my_schemas import *

from datetime import datetime, timedelta, date
import json
# Define the UTC+8 timezone
utc_plus_8 = pytz.timezone('Australia/Perth')  

# Configure destinations of where transformed data should go 
duckdb_dest = dlt.destinations.duckdb(
    credentials="duckdb/warehouse.duckdb"
)

#setup a DuckDB destination -  a local analytics database
parquet_dest = dlt.destinations.filesystem(
    bucket_url="lake/bronze/parquet",
    file_format="parquet"
)

# helper function to convert pyarrow schema to dictionary of dicts + convert data types to DLT compatible
def pyarrow_schema_to_dlt_columns(schema: pa.Schema) -> dict:
    # Mapping from PyArrow types to DLT-compatible types
    type_map = {
        "int64": "bigint",
        "string": "text",
        "float64": "double",
        "bool": "bool",
        "date32": "date",
        "timestamp[us]": "timestamp",#### may need to be changed downstream
        "json": "json"
    }

    return {
        field.name: {
            "name": field.name,
            "data_type": type_map.get(str(field.type), "text")  # default to 'text' if unknown
        }
        for field in schema
    }



In [ ]:
#define source pipeline to read raw data from the folder data_raw
@dlt.source(name="retail_bronze") # schema
def retail_source(raw_path: str = "data_raw"):
    # resources are data loaders 
    # When you wrap a function like load_customers() with @dlt.resource, you're telling DLT: "This is a stream of records I want to load into a destination table."
    # customers becomes the destination table, and the @dlt.resource function acts as a data pipeline component that feeds it
    @dlt.resource(
        write_disposition="replace", # overwrite
        columns=pyarrow_schema_to_dlt_columns(customers_schema)  # Use PyArrow schema that is converted to dict # schema grabbed the schema.py file
    )
    def load_customers():
        # Read CSV and yield data will load in the data from the data_raw file 
        print("Loading resource: customers")
        file_path = os.path.join(raw_path, "customers.csv")
        customersdf = pd.read_csv(file_path)
        for record in customersdf.to_dict(orient="records"):
            yield record #Each record is streamed one at a time, allowing DLT to process efficiently and apply schema validation.
            
    @dlt.resource(
        write_disposition="replace", # overwrite
        columns=pyarrow_schema_to_dlt_columns(products_schema)  # Use PyArrow schema that is converted to dict # schema grabbed the schema.py file
    )
    def load_products():
        # Read CSV and yield data will load in the data from the data_raw file 
        print("Loading resource: products")
        file_path = os.path.join(raw_path, "products.csv")
        productsdf = pd.read_csv(file_path)
        for record in productsdf.to_dict(orient="records"):
            yield record #Each record is streamed one at a time, allowing DLT to process efficiently and apply schema validation.
    @dlt.resource(
            write_disposition="replace", # overwrite
            columns=pyarrow_schema_to_dlt_columns(stores_schema)  # Use PyArrow schema that is converted to dict # schema grabbed the schema.py file
        )
    def load_stores():
        # Read CSV and yield data will load in the data from the data_raw file
        print("Loading resource: stores") 
        file_path = os.path.join(raw_path, "stores.csv")
        storesdf = pd.read_csv(file_path)
        for record in storesdf.to_dict(orient="records"):
            yield record #Each record is streamed one at a time, allowing DLT to process efficiently and apply schema validation.
    @dlt.resource(
            write_disposition="replace", # overwrite
            columns=pyarrow_schema_to_dlt_columns(suppliers_schema)  # Use PyArrow schema that is converted to dict # schema grabbed the schema.py file
        )
    def load_suppliers():
        # Read CSV and yield data will load in the data from the data_raw file 
        print("Loading resource: suppliers")
        file_path = os.path.join(raw_path, "suppliers.csv")
        suppliersdf = pd.read_csv(file_path)
        for record in suppliersdf.to_dict(orient="records"):
            yield record #Each record is streamed one at a time, allowing DLT to process efficiently and apply schema validation.

    @dlt.resource( #ordersheader
            write_disposition="append",
            columns=pyarrow_schema_to_dlt_columns(orders_header_schema),
            primary_key="order_id",
            merge_key="order_id" # is there a reason why there is a merge key? If it's just appending then append on the ts watermark
        )

    def load_ordersheader(updated_after=dlt.sources.incremental("order_ts")):
        print("Loading resource: ordersheader")
        file_path = os.path.join(raw_path, "orders_header.csv")
        ordersheaderdf = pd.read_csv(file_path)
        for record in ordersheaderdf.to_dict(orient="records"):
            if updated_after.last_value is None or record["order_ts"] > updated_after.last_value:
                    yield record  
            

    @dlt.resource( #orderslines
            write_disposition="append",
            columns=pyarrow_schema_to_dlt_columns(orders_lines_schema),
            primary_key=["order_id", "line_number"],
            merge_key=["order_id", "line_number"]
        )


    def load_orderslines(updated_after=dlt.sources.incremental("order_id")):
        print("Loading resource: orderslines")
        file_path = os.path.join(raw_path, "orders_lines.csv")
        orderslinesdf = pd.read_csv(file_path)
        for record in orderslinesdf.to_dict(orient="records"):
            if updated_after.last_value is None or record["order_id"] > updated_after.last_value: #it's monotomically increasing
                yield record


    @dlt.resource(#events
        write_disposition="append",
        primary_key="event_id",
        columns=pyarrow_schema_to_dlt_columns(events_schema)
    )
    def load_events(updated_after=dlt.sources.incremental("envelope.event_ts")):
        print("Loading resource: events")       
        file_path = os.path.join(raw_path, "events.jsonl")  # single file
        with open(file_path, "r") as f:
            for line in f:
                record = json.loads(line)
                if updated_after.last_value is None or record["envelope"]["event_ts"] > updated_after.last_value: 
                    record["event_id"] = record["envelope"]["event_id"]  # flatten for primary key
                    yield record

    @dlt.resource( #sensors
            write_disposition="append",
            columns=pyarrow_schema_to_dlt_columns(sensors_schema)
        )

    def load_sensors(updated_after=dlt.sources.incremental("sensors_ts")):
        print("Loading resource: Sensors")
        file_path = os.path.join(raw_path, "sensors.csv")
        sensorsdf = pd.read_csv(file_path)
        for record in sensorsdf.to_dict(orient="records"):
            if updated_after.last_value is None or record["sensor_ts"] > updated_after.last_value:
                    yield record  

    @dlt.transformer(data_from=load_customers, write_disposition="replace")
    def customers(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }

    @dlt.transformer(data_from=load_products, write_disposition="replace")
    def products(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    @dlt.transformer(data_from=load_stores, write_disposition="replace")
    def stores(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    @dlt.transformer(data_from=load_suppliers, write_disposition="replace")
    def suppliers(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    
    @dlt.transformer(data_from=load_ordersheader, write_disposition="append")
    def ordersheader(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    @dlt.transformer(data_from=load_orderslines, write_disposition="append")
    def orderslines(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    @dlt.transformer(data_from=load_events, write_disposition="append")
    def events(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    @dlt.transformer(data_from=load_sensors, write_disposition="append")
    def sensors(record):
        return {
            **record,
            "ingestion_ts": datetime.now(utc_plus_8),
            "src_filename": dlt.current.source_state().get("file")
        }
    
    return [
        customers,
        products,
        stores,
        suppliers,
        ordersheader,
        orderslines,
        events,
        sensors
    ]


In [4]:
pipelinepq = dlt.pipeline(
    pipeline_name="retail_bronze",
    destination=parquet_dest,
    dataset_name="retail_bronze_dataset"
)


##pipelineduck.drop()  # Clears previous format and schema
##infoduck = pipelineduck.run(retail_source())
pipelinepq.drop()  # Clears previous format and schema
infopq = pipelinepq.run(retail_source(), loader_file_format="parquet") # have to specify the file format here as parquet for some reason

##print(infoduck)
print(infopq)

print( "it's ran")

Loading resource: customers


PipelineStepFailed: Pipeline execution failed at `step=extract` when processing package with `load_id=1758764112.2603588` with exception:

<class 'dlt.extract.exceptions.ResourceExtractionError'>
In processing pipe `load_customers`: extraction of resource `load_customers` in `generator` `load_customers` caused an exception: [Errno 2] No such file or directory: 'data_raw\\customers.csv'